# 🔁 Notebook 4: BiLSTM Multi-Task Classification

**Architecture:** Bidirectional LSTM with Task-Specific Multi-Head Attention

**Tasks:**
- Task 1: Market Event Classification (20 classes)
- Task 2: Sentiment Analysis (Positive / Negative / Neutral)

**Key techniques:**
- Shared BiLSTM encoder (shared feature learning)
- Separate multi-head self-attention per task (task-specific focus)
- Compared against DistilBERT (Notebook 02) using McNemar test
- fp16 mixed precision + early stopping

In [ ]:
!pip install torch scikit-learn matplotlib seaborn pandas numpy scipy

In [ ]:
# ── Import libraries ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, cohen_kappa_score, matthews_corrcoef
)
from sklearn.model_selection import train_test_split, KFold
from scipy.stats import chi2
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 📂 Step 1: Load Data & Encode Labels

In [ ]:
# ── Load cleaned dataset ─────────────────────────────────────────────────────
df = pd.read_csv("data/processed/financial_news_clean.csv")
print(f"Dataset loaded: {df.shape}")

# ── Encode both task labels ───────────────────────────────────────────────────
le_event     = LabelEncoder()
le_sentiment = LabelEncoder()
df["event_label"]     = le_event.fit_transform(df["Market_Event"])
df["sentiment_label"] = le_sentiment.fit_transform(df["Sentiment"])

NUM_EVENTS     = len(le_event.classes_)
NUM_SENTIMENTS = len(le_sentiment.classes_)
print(f"Market Event classes ({NUM_EVENTS}): {le_event.classes_.tolist()}")
print(f"Sentiment classes ({NUM_SENTIMENTS}): {le_sentiment.classes_.tolist()}")

# ── Stratified 70/15/15 split ─────────────────────────────────────────────────
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["event_label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, stratify=temp_df["event_label"], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 🔤 Step 2: Build Vocabulary

In [ ]:
def tokenize(text):
    """Lowercase, remove punctuation, split on whitespace."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return text.split()

# ── Build vocabulary from training data only ──────────────────────────────────
all_tokens = [token for h in train_df["Headline"] for token in tokenize(str(h))]
counter    = Counter(all_tokens)
vocab      = {w: i + 2 for i, (w, _) in enumerate(counter.most_common(30000))}
vocab["<PAD>"] = 0  # padding index
vocab["<UNK>"] = 1  # unknown words
VOCAB_SIZE = len(vocab)
MAX_LEN    = 64
print(f"Vocabulary size: {VOCAB_SIZE} | Max sequence length: {MAX_LEN}")

def encode(text, max_len=MAX_LEN):
    """Tokenize headline and convert to padded token ID list."""
    tokens = tokenize(str(text))[:max_len]
    ids    = [vocab.get(t, 1) for t in tokens]  # 1 = <UNK>
    ids   += [0] * (max_len - len(ids))          # 0 = <PAD>
    return ids

## 🏗️ Step 3: Dataset Class

In [ ]:
class FinancialDataset(Dataset):
    """
    Dataset for BiLSTM multi-task learning.
    Returns encoded headline + both task labels (event + sentiment).
    """
    def __init__(self, df):
        # Encode all headlines to token ID tensors
        self.X = torch.tensor(
            [encode(h) for h in df["Headline"]], dtype=torch.long
        )
        self.event_labels     = torch.tensor(df["event_label"].values, dtype=torch.long)
        self.sentiment_labels = torch.tensor(df["sentiment_label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.event_labels)

    def __getitem__(self, idx):
        return self.X[idx], self.event_labels[idx], self.sentiment_labels[idx]

## 🧠 Step 4: BiLSTM Multi-Task Model

In [ ]:
class TaskSpecificAttention(nn.Module):
    """
    Multi-Head Self-Attention for task-specific feature extraction.

    Why task-specific attention?
    - Market Event detection may focus on financial jargon ("merger", "IPO", "Fed")
    - Sentiment analysis may focus on emotional words ("soar", "crash", "record")
    - Separate attention heads let each task focus on relevant parts independently
    """
    def __init__(self, hidden_size, num_heads=4):
        super().__init__()
        # PyTorch MultiheadAttention: each head learns different aspects
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size * 2,  # BiLSTM output dim
            num_heads=num_heads,
            batch_first=True,
            dropout=0.1
        )

    def forward(self, lstm_output):
        """
        Self-attention: query=key=value=lstm_output
        Returns mean-pooled attended representation.
        """
        attended, _ = self.attn(lstm_output, lstm_output, lstm_output)
        return attended.mean(dim=1)  # mean pool over sequence


class BiLSTMMultiTask(nn.Module):
    """
    Bidirectional LSTM with Shared Encoder + Task-Specific Attention Heads.

    Architecture:
        Headline tokens
            |
        Embedding (vocab -> 300-dim)
            |
        Shared BiLSTM encoder (captures sequential dependencies in both directions)
            |               |
        Event Attention  Sentiment Attention   <- task-specific multi-head attention
            |               |
        Event Head       Sentiment Head        <- task-specific classifiers
        (20 classes)     (3 classes)

    Why shared encoder?
        Both tasks benefit from the same language understanding
        (knowing "Fed raises rates" is useful for both event type AND sentiment)
    """
    def __init__(self, vocab_size, embed_dim=300, hidden_size=128,
                 num_event_classes=20, num_sentiment_classes=3, dropout=0.3):
        super().__init__()

        # Shared embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Shared BiLSTM encoder: processes sequence in both directions
        # hidden_size*2 output because of bidirectional concatenation
        self.lstm = nn.LSTM(
            embed_dim, hidden_size,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
            num_layers=2
        )

        H = hidden_size * 2  # 256 (128 forward + 128 backward)

        # Task-specific multi-head self-attention (separate per task)
        self.event_attention     = TaskSpecificAttention(hidden_size, num_heads=4)
        self.sentiment_attention = TaskSpecificAttention(hidden_size, num_heads=4)

        # Task-specific classification heads
        self.event_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H, num_event_classes)
        )
        self.sentiment_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(H, num_sentiment_classes)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len) token IDs
        Returns:
            event_logits    : (batch, 20)
            sentiment_logits: (batch, 3)
        """
        # Step 1: Embed tokens
        emb = self.dropout(self.embedding(x))        # (B, T, embed_dim)

        # Step 2: Shared BiLSTM encoding
        lstm_out, _ = self.lstm(emb)                 # (B, T, H)

        # Step 3: Task-specific attention
        event_ctx     = self.event_attention(lstm_out)      # (B, H)
        sentiment_ctx = self.sentiment_attention(lstm_out)  # (B, H)

        # Step 4: Classification
        event_logits     = self.event_head(event_ctx)         # (B, 20)
        sentiment_logits = self.sentiment_head(sentiment_ctx) # (B, 3)

        return event_logits, sentiment_logits

## 🚀 Step 5: Training

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE = 32
EPOCHS     = 20
PATIENCE   = 3

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(FinancialDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(FinancialDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(FinancialDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False)

# ── Model, optimizer, loss ────────────────────────────────────────────────────
model = BiLSTMMultiTask(
    VOCAB_SIZE,
    num_event_classes=NUM_EVENTS,
    num_sentiment_classes=NUM_SENTIMENTS
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# ReduceLROnPlateau: reduce LR when val F1 stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5, verbose=True
)
criterion = nn.CrossEntropyLoss()
scaler    = torch.cuda.amp.GradScaler()  # fp16

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_one_epoch(model, loader):
    """Train for one epoch. Returns average loss."""
    model.train()
    total_loss = 0.0
    for X, e_labels, s_labels in loader:
        X        = X.to(DEVICE)
        e_labels = e_labels.to(DEVICE)
        s_labels = s_labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            e_logits, s_logits = model(X)
            # Equal weighting for both tasks
            # Adjust weights (e.g. 0.6/0.4) if one task is more important
            loss = (0.5 * criterion(e_logits, e_labels) +
                    0.5 * criterion(s_logits, s_labels))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    """Evaluate on val/test. Returns F1 scores and raw predictions."""
    model.eval()
    e_preds, e_true = [], []
    s_preds, s_true = [], []
    with torch.no_grad():
        for X, e_labels, s_labels in loader:
            X = X.to(DEVICE)
            e_logits, s_logits = model(X)
            e_preds.extend(e_logits.argmax(1).cpu().numpy())
            e_true.extend(e_labels.numpy())
            s_preds.extend(s_logits.argmax(1).cpu().numpy())
            s_true.extend(s_labels.numpy())
    e_f1 = f1_score(e_true, e_preds, average="macro")
    s_f1 = f1_score(s_true, s_preds, average="macro")
    return e_f1, s_f1, e_preds, e_true, s_preds, s_true


# ── Training loop with early stopping ────────────────────────────────────────
best_val_f1      = 0.0
patience_counter = 0
train_losses     = []
val_f1_history   = []

print(f"{'Epoch':<8} {'Loss':<12} {'Event F1':<14} {'Sent F1':<12} {'Avg F1'}")
print("-" * 55)

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader)
    e_f1, s_f1, _, _, _, _ = evaluate(model, val_loader)
    avg_f1 = (e_f1 + s_f1) / 2

    train_losses.append(loss)
    val_f1_history.append(avg_f1)
    scheduler.step(avg_f1)

    print(f"{epoch+1:<8} {loss:<12.4f} {e_f1:<14.4f} {s_f1:<12.4f} {avg_f1:.4f}")

    if avg_f1 > best_val_f1:
        best_val_f1 = avg_f1
        torch.save(model.state_dict(), "models/saved/bilstm_best.pt")
        patience_counter = 0
        print(f"         -> Best model saved! (F1={best_val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"
Early stopping at epoch {epoch+1}.")
            break

print(f"
Best Validation F1: {best_val_f1:.4f}")

## 📊 Step 6: Evaluation

In [ ]:
# ── Load best model and run test evaluation ───────────────────────────────────
model.load_state_dict(torch.load("models/saved/bilstm_best.pt"))
e_f1, s_f1, e_preds, e_true, s_preds, s_true = evaluate(model, test_loader)

print("=" * 60)
print("TASK 1: Market Event — Test Set Results")
print("=" * 60)
print(classification_report(e_true, e_preds, target_names=le_event.classes_))
print(f"Cohen Kappa : {cohen_kappa_score(e_true, e_preds):.4f}")
print(f"MCC         : {matthews_corrcoef(e_true, e_preds):.4f}")

print("
" + "=" * 60)
print("TASK 2: Sentiment — Test Set Results")
print("=" * 60)
print(classification_report(s_true, s_preds, target_names=le_sentiment.classes_))
print(f"Cohen Kappa : {cohen_kappa_score(s_true, s_preds):.4f}")
print(f"MCC         : {matthews_corrcoef(s_true, s_preds):.4f}")

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle("BiLSTM Multi-Task — Confusion Matrices", fontsize=14, fontweight="bold")

sns.heatmap(confusion_matrix(e_true, e_preds), annot=True, fmt="d", cmap="Purples",
            xticklabels=le_event.classes_, yticklabels=le_event.classes_, ax=axes[0])
axes[0].set_title("Market Event")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].tick_params(axis="x", rotation=90)

sns.heatmap(confusion_matrix(s_true, s_preds), annot=True, fmt="d", cmap="Oranges",
            xticklabels=le_sentiment.classes_, yticklabels=le_sentiment.classes_, ax=axes[1])
axes[1].set_title("Sentiment")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig("results/plots/bilstm_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 🆚 Step 7: Model Comparison — BiLSTM vs DistilBERT

In [ ]:
# ── McNemar Test: statistical significance of performance difference ─────────
# McNemar test checks if two classifiers make significantly different errors
# Null hypothesis: both models make the same errors
# p < 0.05 = statistically significant difference

def mcnemar_test(preds_a, preds_b, true_labels):
    """
    Compute McNemar test statistic between two models.
    n01 = model A correct, model B wrong
    n10 = model A wrong,   model B correct
    """
    n01 = sum(1 for a, b, t in zip(preds_a, preds_b, true_labels)
              if a == t and b != t)
    n10 = sum(1 for a, b, t in zip(preds_a, preds_b, true_labels)
              if a != t and b == t)

    if (n01 + n10) == 0:
        return 0, 1.0  # identical predictions

    # Chi-squared statistic with continuity correction
    statistic = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p_value   = 1 - chi2.cdf(statistic, df=1)
    return statistic, p_value

print("McNemar Test: BiLSTM vs DistilBERT (Market Event)")
print("NOTE: Load DistilBERT predictions from Notebook 02 to use this test.")
print("Example usage:")
print("  stat, p = mcnemar_test(bilstm_event_preds, distilbert_event_preds, e_true)")
print("  print(f'p-value: {p:.4f} -> significant: {p < 0.05}')")

In [ ]:
# ── 5-Fold Cross Validation on full dataset ───────────────────────────────────
# More robust evaluation — tests model across different data splits
# Reports mean +/- std of F1 score

print("Running 5-Fold Cross Validation on BiLSTM...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_e_f1s, fold_s_f1s = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    fold_train = df.iloc[train_idx].reset_index(drop=True)
    fold_val   = df.iloc[val_idx].reset_index(drop=True)

    fold_train_loader = DataLoader(FinancialDataset(fold_train),
                                   batch_size=BATCH_SIZE, shuffle=True)
    fold_val_loader   = DataLoader(FinancialDataset(fold_val),
                                   batch_size=BATCH_SIZE, shuffle=False)

    # Fresh model for each fold
    fold_model = BiLSTMMultiTask(VOCAB_SIZE, num_event_classes=NUM_EVENTS,
                                  num_sentiment_classes=NUM_SENTIMENTS).to(DEVICE)
    fold_opt   = torch.optim.Adam(fold_model.parameters(), lr=1e-3)

    # Train for 5 epochs per fold (reduce for speed)
    for _ in range(5):
        fold_model.train()
        for X, el, sl in fold_train_loader:
            X, el, sl = X.to(DEVICE), el.to(DEVICE), sl.to(DEVICE)
            fold_opt.zero_grad()
            elo, slo = fold_model(X)
            loss = 0.5 * criterion(elo, el) + 0.5 * criterion(slo, sl)
            loss.backward()
            fold_opt.step()

    ef1, sf1, _, _, _, _ = evaluate(fold_model, fold_val_loader)
    fold_e_f1s.append(ef1)
    fold_s_f1s.append(sf1)
    print(f"  Fold {fold+1} | Event F1: {ef1:.4f} | Sentiment F1: {sf1:.4f}")

print(f"
Event F1     : {np.mean(fold_e_f1s):.4f} +/- {np.std(fold_e_f1s):.4f}")
print(f"Sentiment F1 : {np.mean(fold_s_f1s):.4f} +/- {np.std(fold_s_f1s):.4f}")

In [ ]:
# ── Save final metrics ────────────────────────────────────────────────────────
metrics = {
    "Model"              : "BiLSTM",
    "Event_Macro_F1"     : round(f1_score(e_true, e_preds, average="macro"), 4),
    "Event_Weighted_F1"  : round(f1_score(e_true, e_preds, average="weighted"), 4),
    "Event_Kappa"        : round(cohen_kappa_score(e_true, e_preds), 4),
    "Event_MCC"          : round(matthews_corrcoef(e_true, e_preds), 4),
    "Event_CV_Mean_F1"   : round(np.mean(fold_e_f1s), 4),
    "Event_CV_Std_F1"    : round(np.std(fold_e_f1s), 4),
    "Sent_Macro_F1"      : round(f1_score(s_true, s_preds, average="macro"), 4),
    "Sent_Weighted_F1"   : round(f1_score(s_true, s_preds, average="weighted"), 4),
    "Sent_Kappa"         : round(cohen_kappa_score(s_true, s_preds), 4),
    "Sent_MCC"           : round(matthews_corrcoef(s_true, s_preds), 4),
    "Sent_CV_Mean_F1"    : round(np.mean(fold_s_f1s), 4),
    "Sent_CV_Std_F1"     : round(np.std(fold_s_f1s), 4),
}

pd.DataFrame([metrics]).to_csv("results/metrics/bilstm_metrics.csv", index=False)
print("Metrics saved to results/metrics/bilstm_metrics.csv")
print(pd.DataFrame([metrics]).T)